# Render Remotion via Google Colab GPU

Notebook ini mengambil ZIP project dari repository GitHub, memasang dependency Remotion, lalu merender MP4 dengan Chromium GPU. Pilih runtime GPU di **Runtime → Change runtime type → T4 GPU** sebelum menjalankan semua cell.

In [ ]:
# @title Konfigurasi
PROJECT = "peter-episode1" # @param {type:"string"}
COMPOSITION = "" # @param {type:"string"}
REPO = "idinlabs-dev/render-remotion" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}
OUTPUT_NAME = "" # @param {type:"string"}


In [ ]:
# @title Ambil GitHub token dari Colab Secrets
from google.colab import userdata
import os
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("Buat Colab Secret bernama GITHUB_TOKEN berisi Personal Access Token GitHub dengan akses Contents read.")
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
print("Token GitHub tersedia")


In [ ]:
# @title Install system packages dan Node.js
!sudo apt-get update -qq
!sudo apt-get install -y -qq ffmpeg libnss3 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libasound2t64 libxrandr2 libxkbcommon-x11-0 libxfixes3 libxcomposite1 libxdamage1 libgbm-dev libcups2 libpango-1.0-0 libcairo2 unzip curl
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
!sudo apt-get install -y -qq nodejs
!node --version
!npm --version


In [ ]:
# @title Download ZIP project dari GitHub
import requests, zipfile, io, pathlib, shutil, os
root = pathlib.Path("/content/remotion-render")
if root.exists(): shutil.rmtree(root)
root.mkdir(parents=True)
url = f"https://api.github.com/repos/{REPO}/contents/projects/{PROJECT}.zip?ref={BRANCH}"
r = requests.get(url, headers={"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept":"application/vnd.github.raw"})
r.raise_for_status()
archive = root / f"{PROJECT}.zip"
archive.write_bytes(r.content)
extract = root / "project"
extract.mkdir()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall(extract)
if not (extract/"package.json").exists():
    children=[p for p in extract.iterdir() if p.name != "__MACOSX"]
    if len(children)==1 and children[0].is_dir() and (children[0]/"package.json").exists():
        extract=children[0]
if not (extract/"package.json").exists():
    raise RuntimeError("package.json tidak ditemukan dalam ZIP")
print("Project:", extract)


In [ ]:
# @title Install dependency Remotion
%cd {extract}
!npm install --no-audit --no-fund
!npx remotion browser ensure


In [ ]:
# @title Deteksi entry dan composition
import json, subprocess, shlex, os
cfg={}
if pathlib.Path("render.json").exists(): cfg=json.loads(pathlib.Path("render.json").read_text())
entry=cfg.get("entry", "")
if not entry:
    for candidate in ["src/index.ts","src/index.tsx","src/index.js","src/index.jsx"]:
        if pathlib.Path(candidate).exists(): entry=candidate; break
if not entry: raise RuntimeError("Entry Remotion tidak ditemukan")
composition=COMPOSITION or cfg.get("composition", "")
ids=subprocess.check_output(["npx","remotion","compositions",entry,"--quiet"], text=True).strip().splitlines()[-1].split()
if not composition: composition=ids[0]
if composition not in ids: raise RuntimeError(f"Composition {composition} tidak tersedia. Pilihan: {ids}")
print("Entry:",entry)
print("Composition:",composition)


In [ ]:
# @title Render MP4 dengan GPU Chromium
out=pathlib.Path("out")
out.mkdir(exist_ok=True)
name=OUTPUT_NAME or f"{PROJECT}.mp4"
outfile=out/name
cmd=["npx","remotion","render",entry,composition,str(outfile),"--codec=h264","--gl=angle","--concurrency=2"]
print(" ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
print("Selesai:", outfile, "bytes:", outfile.stat().st_size)


In [ ]:
# @title Download hasil render
from google.colab import files
files.download(str(outfile))


In [ ]:
# @title (Opsional) Upload MP4 ke GitHub Release
# Jalankan cell ini jika ingin menyimpan hasil ke Release GitHub.
import time
tag=f"colab-{PROJECT}-{int(time.time())}"
headers={"Authorization":f"Bearer {GITHUB_TOKEN}","Accept":"application/vnd.github+json"}
release=requests.post(f"https://api.github.com/repos/{REPO}/releases",headers=headers,json={"tag_name":tag,"name":f"Colab render {PROJECT}","draft":False,"prerelease":False})
release.raise_for_status(); upload_url=release.json()["upload_url"].split("{")[0]
with open(outfile,"rb") as fh:
    up=requests.post(upload_url,headers={**headers,"Content-Type":"video/mp4"},params={"name":name},data=fh)
up.raise_for_status()
print("Release:",release.json()["html_url"])
